# BizLens v2.2.12 - Q-Learning (Progressive Levels)

Learn Reinforcement Learning step-by-step: **Beginner → Intermediate → Advanced**.
We solve a Grid World problem using BizLens for analysis.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bizlens"])
print("✅ BizLens v2.2.13 ready for Progressive Q-Learning!")

In [ ]:
import bizlens as bl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
# bl.set_profiling(True)

# ── Optional: Use a BizLens built-in dataset ─────────────────────────────
# event_log = bl.generate_tech_support_event_log()  # model ticket routing as states/actions
# event_log = bl.generate_manufacturing_event_log()  # model production steps as RL environment
#
# After loading, explore with:
# print(df.shape)         # rows × columns
# print(df.dtypes)        # column types
# print(df.head())        # first 5 rows
#
# ── Export / Save Results ─────────────────────────────────────────────────
# import numpy as np; np.save('q_table.npy', Q)  # Save trained Q-table
# df.to_excel('output.xlsx', index=False)  # Save as Excel
# df.to_json('output.json', orient='records', indent=2)  # Save as JSON

## Environment: Grid World

In [ ]:
class GridWorld:
    def __init__(self):
        self.size = 5
        self.start = (0, 0)
        self.goal = (4, 4)
        self.obstacles = [(1,1), (2,2), (3,1)]
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        x, y = self.state
        if action == 0: x = max(0, x-1)      # Up
        elif action == 1: x = min(4, x+1)    # Down
        elif action == 2: y = max(0, y-1)    # Left
        elif action == 3: y = min(4, y+1)    # Right
        self.state = (x, y)
        
        if self.state == self.goal:
            return self.state, 100, True
        elif self.state in self.obstacles:
            return self.state, -50, True
        return self.state, -1, False

env = GridWorld()
print("Grid World environment ready.")

## Level 1: Beginner – Basic Q-Learning

In [ ]:
def q_learning_basic(env, episodes=800, alpha=0.1, gamma=0.9, epsilon=0.2):
    Q = np.zeros((5, 5, 4))
    rewards = []
    
    for ep in range(episodes):
        state = env.reset()
        total_r = 0
        done = False
        while not done:
            if np.random.rand() < epsilon:
                action = np.random.randint(0, 4)
            else:
                action = np.argmax(Q[state[0], state[1]])
            next_state, reward, done = env.step(action)
            total_r += reward
            best_next = np.max(Q[next_state[0], next_state[1]])
            Q[state[0], state[1], action] += alpha * (reward + gamma * best_next - Q[state[0], state[1], action])
            state = next_state
        rewards.append(total_r)
    return Q, rewards

Q_basic, rewards_basic = q_learning_basic(env)
print("Basic Q-Learning training completed.")

In [ ]:
plt.plot(rewards_basic, alpha=0.7)
plt.title('Level 1: Basic Q-Learning - Learning Curve')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.show()

## Level 2: Intermediate – Epsilon Decay + Better Policy

In [ ]:
def q_learning_intermediate(env, episodes=1500, alpha=0.1, gamma=0.95):
    Q = np.zeros((5, 5, 4))
    rewards = []
    epsilon = 1.0
    
    for ep in range(episodes):
        state = env.reset()
        total_r = 0
        done = False
        while not done:
            if np.random.rand() < epsilon:
                action = np.random.randint(0, 4)
            else:
                action = np.argmax(Q[state[0], state[1]])
            next_state, reward, done = env.step(action)
            total_r += reward
            best_next = np.max(Q[next_state[0], next_state[1]])
            Q[state[0], state[1], action] += alpha * (reward + gamma * best_next - Q[state[0], state[1], action])
            state = next_state
        rewards.append(total_r)
        epsilon = max(0.01, epsilon * 0.995)  # decay
    return Q, rewards

Q_int, rewards_int = q_learning_intermediate(env)
print("Intermediate Q-Learning training completed.")

In [ ]:
plt.plot(rewards_int, alpha=0.7, label='Intermediate')
plt.plot(rewards_basic, alpha=0.4, label='Basic')
plt.legend()
plt.title('Level 2: Comparison of Learning Curves')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.show()

## Level 3: Advanced – Analysis with BizLens

In [ ]:
# Final Policy
policy = np.argmax(Q_int, axis=2)
actions = ['↑','↓','←','→']
print("Learned Optimal Policy:")
for row in policy:
    print([actions[a] for a in row])

In [ ]:
# BizLens Analysis on Rewards
rewards_series = pd.Series(rewards_int)
bl.tables.summary_statistics(rewards_series.to_frame())
bl.tables.percentile_table(rewards_series)

In [ ]:
# Reward Distribution
plt.figure(figsize=(10,5))
sns.histplot(rewards_int, bins=50, kde=True, color='purple')
plt.title('Final Reward Distribution (Advanced Q-Learning)')
plt.xlabel('Total Reward per Episode')
plt.show()